In [1]:
# 0. Environment Setup

# Clone the new kauldron repository
!git clone https://github.com/google-research/kauldron || true
!pip install -q ./kauldron
# Clone the gemma repository if not already present
!git clone https://github.com/google-deepmind/gemma.git || true
!pip install -q ./gemma

# Clone the dialog repository to fix Gemma format issues
!git clone https://github.com/google-deepmind/dialog || true
!pip install -q ./dialog

!pip install -q flax optax seqio



# Ensure our local modules are in the Python path
import sys
import os
sys.path.append(os.getcwd())

Cloning into 'kauldron'...
remote: Enumerating objects: 9546, done.
remote: Counting objects: 100% (153/153), done.
remote: Compressing objects: 100% (130/130), done.
remote: Total 9546 (delta 41), reused 33 (delta 23), pack-reused 9393 (from 3)
Receiving objects: 100% (9546/9546), 2.84 MiB | 31.91 MiB/s, done.
Resolving deltas: 100% (6861/6861), done.
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 kB 4.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 kB 4.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.7/5.7 MB 117.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.8/101.8 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.3/47.3 

In [2]:
# %rm -rf /content/Titans_jax

In [3]:
!git clone https://github.com/andrew-veriga/Titans_jax.git
%cd Titans_jax

Cloning into 'Titans_jax'...
remote: Enumerating objects: 239, done.
remote: Counting objects: 100% (25/25), done.
remote: Compressing objects: 100% (18/18), done.
remote: Total 239 (delta 12), reused 14 (delta 7), pack-reused 214 (from 1)
Receiving objects: 100% (239/239), 981.49 KiB | 20.45 MiB/s, done.
Resolving deltas: 100% (142/142), done.
/content/Titans_jax


In [4]:
from kauldron import kd
from kauldron.ktyping import Array

/usr/local/lib/python3.12/dist-packages/jax/_src/cloud_tpu_init.py:86: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(


In [5]:
import jax
import jax.numpy as jnp
import optax
from kauldron import kd
import numpy as np
import os
import orbax.checkpoint as ocp

# Gemma imports
from gemma import gm

# Our custom Titans integration
import importlib
import gemma_titans
importlib.reload(gemma_titans)
from gemma_titans import Gemma3_1B_Titans
from titans_ckpts import SkipTitans
import titans_tree_utils

print(f"JAX Backend: {jax.default_backend()}")
print(f"Devices: {jax.devices()}")

# Prevent JAX from allocating 100% of TPU memory instantly
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
# Limit XLA to 85% of TPU HBM to leave room for overhead
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = ".85"
# Reduce fragmentation and compilation memory spike
os.environ["XLA_FLAGS"] = "--xla_gpu_enable_highest_priority_async_collectives=true --xla_tpu_enable_data_parallel_all_reduce_opt=true --xla_tpu_memory_bound_loop_fusion_limit=1"
os.environ["JAX_COMPILATION_CACHE_DIR"] = "/tmp/jax_cache"


JAX Backend: tpu
Devices: [TpuDevice(id=0, process_index=0, coords=(0,0,0), core_on_chip=0)]


## 3. Save / Load Trained Weights
We don't want to save the entire 1B model, just our new memory projections. We utilize the `split_titans_params` utility.

In [9]:
import zipfile
zipfile

<module 'zipfile' from '/usr/lib/python3.12/zipfile/__init__.py'>

In [10]:
import google.colab
import shutil

shutil.unpack_archive('/content/saved_titans_delta.zip',"./saved_titans_delta")

In [11]:
def load_titans_weights(load_dir: str):
    load_path = os.path.abspath(load_dir)
    checkpointer = ocp.StandardCheckpointer()

    # Restore directly without a template (returns dictionary of arrays)
    titans_params = checkpointer.restore(load_path)
    return titans_params

# Usage:

state = load_titans_weights("./saved_titans_delta")

In [12]:
!ls ./saved_titans_delta


array_metadatas       d		      _METADATA        _sharding
_CHECKPOINT_METADATA  manifest.ocdbt  ocdbt.process_0


In [13]:
gemma_config = gm.nn.Gemma3_1B.config

# Initialize model
model = Gemma3_1B_Titans(
    config=gemma_config,
    dtype=jnp.bfloat16,
    return_last_only=False,
    tokens="batch.input",
)
model.titans_layer_indices = [11] # Add Titans memory only to the middle layer

# Load original Gemma 3 1B IT weights
print("Loading original Gemma weights...")
original_params = gm.ckpts.load_params(gm.ckpts.CheckpointPath.GEMMA3_1B_IT)

# Merge loaded Titans weights into original Gemma params
print("Merging Titans weights...")
import titans_tree_utils
loaded_titans_params = load_titans_weights("./saved_titans_delta")
merged_params = titans_tree_utils.merge_titans_params(original_params, loaded_titans_params)

# Create a proper Kauldron dataset pipeline using TFDS
batch_size = 8
max_length = 256

def format_squad(x):
    # SQuAD structure: context, question, answers (dict with 'text' list)
    answers = x.get('answers', {}).get('text', [])
    answer = answers[0] if len(answers) > 0 else "Unknown"

    # Robustly handle potential bytes from TFDS
    ctx = x["context"].decode('utf-8') if hasattr(x["context"], 'decode') else x["context"]
    q = x["question"].decode('utf-8') if hasattr(x["question"], 'decode') else x["question"]
    ans = answer.decode('utf-8') if hasattr(answer, 'decode') else answer

    # Create 'src' and 'dst' for the Seq2SeqTask
    x['src'] = "Context: " + ctx + "\n\nQuestion: " + q
    x['dst'] = ans
    return x

tokenizer = gm.text.Gemma3Tokenizer()


## 4. Dialogue Inference (Test-Time Dynamic Memory)
During inference, the model passes a `cache` containing both the standard Attention KV-cache AND the Titans Neural Memory State. The model automatically updates its internal state.

In [16]:
# We use Gemma's built-in Sampler to correctly handle positions, attention masks, and cache updates
sampler = gm.text.Sampler(
    model=model,
    params=merged_params,
    tokenizer=tokenizer,
)

print("Simulation: User enters a turn...")
prompt = "Context: The Titans are powerful mythological entities.\n\nQuestion: Who are the Titans?"
prompt = "Titans - технология запоминания контекста во время ответа LLM"
# The sampler will automatically prefill the cache, update Titans memory, and generate text
output = sampler.sample(prompt, max_new_tokens=20)

print("\n--- Generated Response ---")
print(output)
print("\nMemory state updated automatically in cache.")

Simulation: User enters a turn...


ScopeParamNotFoundError: Could not find parameter named "input_embedding" in scope "/jit()/embedder". (https://flax.readthedocs.io/en/latest/api_reference/flax.errors.html#flax.errors.ScopeParamNotFoundError)

## 5. (Bonus) Fast Loading via `jax.eval_shape`
In a new session, to avoid wasting time and memory initializing random weights before loading, you can create a zero-memory structural template using `jax.eval_shape`.

In [ ]:
import jax
import jax.numpy as jnp
import orbax.checkpoint as ocp
import titans_tree_utils

# 1. Define model config (must match the training config)
model_eval = Gemma3_1B_Titans(
    config=gm.nn.Gemma3_1B.config,
    dtype=jnp.bfloat16,
    return_last_only=False,
    tokens="batch.input",
)
model_eval.titans_layer_indices = [11]

# 2. MAGIC: Get a "hollow" template without allocating HBM memory
template_variables = jax.eval_shape(
    model_eval.init,
    jax.random.PRNGKey(0),
    tokens=jnp.ones((1, 1), dtype=jnp.int32)
)

# 3. Extract the Titans portion of the template
_, titans_template = titans_tree_utils.split_titans_params(template_variables['params'])

# 4. Load the saved weights using this lightweight template
checkpointer = ocp.StandardCheckpointer()
state = load_titans_weights(state, "./saved_titans_delta")
# print("Titans weights loaded successfully!")
sampler = gm.text.Sampler(
    model=model_eval,
    params=state.params,
    tokenizer=tokenizer,
)

print("Simulation: User enters a turn...")
prompt = "Context: The Titans are powerful mythological entities.\n\nQuestion: Who are the Titans?"
prompt = "Titans - технология запоминания контекста во время ответа LLM"
# The sampler will automatically prefill the cache, update Titans memory, and generate text
sampler.sample(prompt, max_new_tokens=20)

Simulation: User enters a turn...


In [ ]:
sampler.sample(prompt, max_new_tokens=20)

' program down.wordzystava text. туда неavut_perwell, text aftertext로,'